# Snowflake Standalone Notebook
This notebook is self-contained and can run directly in Snowflake Notebooks without external dependencies.

In [ ]:
import os
import snowflake.connector

conn = None
session = None
TABLE_NAME = os.getenv("SNOWFLAKE_TABLE", "TABLE1")

# 1) Try Snowflake Notebook active session
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    print("✅ Using Snowflake active session")
except Exception:
    # 2) Fallback to local .env + connector
    try:
        from dotenv import load_dotenv
        load_dotenv()

        conn = snowflake.connector.connect(
            user=os.getenv("SNOWFLAKE_USER"),
            password=os.getenv("SNOWFLAKE_PASSWORD"),
            account=os.getenv("SNOWFLAKE_ACCOUNT"),
            warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
            database=os.getenv("SNOWFLAKE_DATABASE"),
            schema=os.getenv("SNOWFLAKE_SCHEMA"),
        )
        print("✅ Connected using environment variables")
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        print("Set .env values for local run, or run this notebook inside Snowflake.")

✅ Connected using environment variables


## SELECT Query

In [ ]:
def run_query(query):
    """Execute SELECT query and return results"""
    try:
        if session is not None:
            rows = session.sql(query).collect()
            return rows

        if conn is None:
            print("No connection/session available")
            return None

        cursor = conn.cursor()
        cursor.execute(query)
        results = cursor.fetchall()
        cursor.close()
        return results
    except Exception as e:
        print(f"Error: {e}")
        return None

# Test query
result = run_query("SELECT CURRENT_VERSION();")
print(result)

[('10.7.1',)]


In [ ]:
# Query target table
result = run_query(f"SELECT * FROM {TABLE_NAME} LIMIT 10;")
if result:
    for row in result:
        print(row)

## End-to-End Flow: INSERT → SELECT → UPDATE → SELECT → DELETE
This cell runs the exact sequence on one test row.

In [ ]:
def execute_dml(query):
    """Execute INSERT/UPDATE/DELETE queries"""
    try:
        if session is not None:
            session.sql(query).collect()
            return True

        if conn is None:
            print("No connection/session available")
            return False

        cursor = conn.cursor()
        cursor.execute(query)
        conn.commit()
        cursor.close()
        return True
    except Exception as e:
        print(f"Error: {e}")
        return False


# Test row id (use unique-ish value to avoid collisions)
import time
TEST_ID = int(time.time()) % 1000000

print(f"Running flow on {TABLE_NAME} with ID={TEST_ID}")

# 1) INSERT
insert_query = f"INSERT INTO {TABLE_NAME} (ID, NAME) VALUES ({TEST_ID}, 'Notebook Insert');"
print("\n1) INSERT")
print("✅ Insert successful" if execute_dml(insert_query) else "❌ Insert failed")

# 2) SELECT (after insert)
print("\n2) SELECT after insert")
rows = run_query(f"SELECT * FROM {TABLE_NAME} WHERE ID = {TEST_ID};")
print(rows)

# 3) UPDATE
update_query = f"UPDATE {TABLE_NAME} SET NAME = 'Notebook Updated' WHERE ID = {TEST_ID};"
print("\n3) UPDATE")
print("✅ Update successful" if execute_dml(update_query) else "❌ Update failed")

# 4) SELECT (after update)
print("\n4) SELECT after update")
rows = run_query(f"SELECT * FROM {TABLE_NAME} WHERE ID = {TEST_ID};")
print(rows)

# 5) DELETE
delete_query = f"DELETE FROM {TABLE_NAME} WHERE ID = {TEST_ID};"
print("\n5) DELETE")
print("✅ Delete successful" if execute_dml(delete_query) else "❌ Delete failed")

✅ Insert successful
✅ Update successful
✅ Delete successful


In [ ]:
# Cleanup (for local connector mode)
try:
    if conn is not None:
        conn.close()
        print("Connection closed")
except Exception:
    pass